In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [19]:
class FraudDetector:
    def __init__(self, contamination='auto', random_state=30):
        self.contamination = contamination
        self.random_state = random_state
        self.model = None
        self.feature_names = None
        self.threshold = None
        self._initialize_pipeline()

    def _initialize_pipeline(self):
        numeric_features = [
            'amount', 'credit_score',
             'transaction_velocity', 'amount_deviation',"account_age_years","avg_monthly_income","risk_score"
        ]
        categorical_features = ['transaction_type', 'location','time_of_day','auth_method','age_group','home_location','account_type','mobile_banking_user','employment_status','international_activity']

        self.preprocessor = ColumnTransformer(transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])

        self.model = Pipeline([
            ('preprocessor', self.preprocessor),
            ('detector', IsolationForest(
                n_estimators=200,
                contamination=self.contamination,
                max_features=1,
                random_state=self.random_state,
                verbose=1
            ))
        ])

    def engineer_features(self, df):
        df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

        # Print the hour part of the 'transaction_date'
        df["hour"]=df["transaction_date"].dt.hour
        df = df.fillna(df.mean(numeric_only=True))
        return df

    def fit(self, df):
        df = self.engineer_features(df.copy())
        self.model.fit(df)
        scores = self.model.named_steps['detector'].score_samples(
            self.model.named_steps['preprocessor'].transform(df)
        )
        self.threshold = np.percentile(scores, 100 * float(self.contamination))

        num_features = self.preprocessor.transformers_[0][2]
        cat_features = self.preprocessor.named_transformers_['cat'].get_feature_names_out()
        self.feature_names = num_features + list(cat_features)

    def predict(self, df):
        if not self.model:
            raise RuntimeError("Model not trained.")

        df = self.engineer_features(df.copy())
        transformed = self.model.named_steps['preprocessor'].transform(df)
        df['anomaly_score'] = self.model.named_steps['detector'].score_samples(transformed)
        df['predictedFraud'] = (df['anomaly_score'] < self.threshold).astype(int)
        return df

    def evaluate(self, df):
        if "isFraud" not in df.columns:
            raise ValueError("Data needs an 'isFraud' column for evaluation.")

        preds = self.predict(df)
        print(classification_report(preds['isFraud'], preds['predictedFraud']))

        cm = confusion_matrix(preds['isFraud'], preds['predictedFraud'])
        sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.show()

    def feature_importance(self):
        if not hasattr(self.model.named_steps['detector'], 'feature_importances_'):
            print("Feature importance is not available for this model.")
            return

        plt.figure(figsize=(12, 8))
        pd.Series(
            self.model.named_steps['detector'].feature_importances_,
            index=self.feature_names
        ).sort_values().plot.barh()
        plt.title("Fraud Detection Feature Importance")
        plt.show()

    def save(self, path):
        joblib.dump({
            "model": self.model,
            "threshold": self.threshold,
            "features": self.feature_names
        }, path)

    @classmethod
    def load(cls, path):
        data = joblib.load(path)
        detector = cls()
        detector.model = data['model']
        detector.threshold = data['threshold']
        detector.feature_names = data['features']
        return detector



In [20]:
if __name__ == "__main__":
    dfu = pd.read_csv("/Users/lalitramanmishra/GlobalIME/global_ime_bank_customers 2.csv")
    for customer_id in dfu["customer_id"].unique():
        print("Generating transaction data")
        transactions = pd.read_csv(f"/Users/lalitramanmishra/GlobalIME/segmented_transactions/user_{customer_id}_transactions.csv")

        detector = FraudDetector(contamination=0.06531)
        print("Training fraud detection model...")


        # Train the model on the entire dataset
        detector.fit(transactions)

        # Predict anomalies on the same dataset
        
    
        results = detector.predict(transactions)
        print("\nFraud Detection Results:")
        print(results[results['predictedFraud'] == 1][['amount', 'anomaly_score']])

        # Save the trained model
        detector.save(f'models/fraud_detector{customer_id}.joblib')
        print("Model saved successfully")
        

Generating transaction data
Training fraud detection model...


ValueError: X should be in csr_matrix format, got <class 'scipy.sparse._csc.csc_matrix'>